## GPT Pilot Regeneration — Prompt v3

This notebook is a pilot and does not create the final synthetic dataset. Earlier v2 execution cells are archived below and disabled; their recorded outputs are historical. MOCK PILOT, REAL GROQ GPT PILOT v2 and this v3 regeneration remain separate. No Llama generation, training, splitting or dataset merging is performed.

In [1]:
from pathlib import Path
import json
import os
import sys
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'src/regeneration_v3.py').exists()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
v2_path = ROOT / 'data/synthetic/pilot/groq_gpt/openai_gpt-oss-120b'
print('Frozen v2 pilot summary:')
display(pd.read_csv(v2_path / 'summary.csv'))

Frozen v2 pilot summary:


,metric,value
0,number_selected,20
1,number_attempted,20
2,number_successfully_generated,13
3,number_generation_failures,7
4,number_not_attempted,0
5,number_structurally_valid,13
6,number_automatically_passing,3
7,number_automatically_rejected,7
8,number_requiring_human_review,13
9,number_with_completed_human_review,0


### Reason for Regeneration

The AI-assisted review classified 13 successful v2 generations as 1 provisionally suitable, 8 requiring revision and 4 rejected. This is not completed human validation. Problems included research artifacts, unsupported facts and excessive use of upfront fees. Source 10308 is excluded. The same twelve specified legitimate source records are reused, with two assignments to each of six mechanism categories; assignments are not evidence of successful expression of those mechanisms.

In [2]:
v3_output = ROOT / 'data/synthetic/pilot/groq_gpt_v3'
v3_manifest = json.loads((v3_output / 'manifest.json').read_text(encoding='utf-8'))
v3_summary = json.loads((v3_output / 'generation_summary.json').read_text(encoding='utf-8'))
print('Frozen v3 artifact - read only')
display({key: v3_manifest[key] for key in ['provider', 'model', 'prompt_version', 'validation_version', 'source_ids', 'excluded_source_id']})

Frozen v3 artifact - read only


{'provider': 'groq',
 'model': 'openai/gpt-oss-120b',
 'prompt_version': 'v3',
 'validation_version': 'v3',
 'source_ids': [11662,
  12066,
  13604,
  14350,
  14484,
  15885,
  3203,
  4581,
  5581,
  6199,
  7192,
  9208],
 'excluded_source_id': 10308}

### Validation Criteria and Source Fidelity

Strict five-field JSON, non-empty title/description, source-specific role/responsibility/qualification/location anchors, empty-field preservation, financial and employment fact checks, identifier removal, context-sensitive artifact detection, recruitment-specific fraud evidence and integrated transformation are required. Lexical similarity is only an auditable proxy, not proof of semantic preservation. Unsupported-fact checks are conservative heuristics and cannot exhaustively establish factual equivalence. Every candidate still requires independent human review. No criteria are relaxed after observing the smoke result.

### Model Availability and One-Record Smoke Test

Only source 11662 is attempted first. Any failure of the complete v3 validation stops the remaining eleven. Calls are sequential, at least 65 seconds apart, with up to three attempts per source; Retry-After is honored and repeated rate limits stop the run. A saved attempt is never automatically regenerated.

In [3]:
v3_state = json.loads((v3_output / 'state.json').read_text(encoding='utf-8'))
print('Frozen v3 smoke status:', v3_state['smoke_status'])
print('Stop reason:', v3_state['stop_reason'])

Frozen v3 smoke status: failed
Stop reason: Smoke candidate failed full v3 validation; remaining 11 not generated.


### Remaining Eleven and Separate Exports

Continuation is conditional on the full smoke test passing. Saved state prevents duplicate calls on rerun. Failed validation is preserved, not silently repaired. Original v2 and EMSCAD files are checked against frozen hashes.

In [4]:
display(v3_summary)
display(pd.read_csv(v3_output / 'validation_results.csv'))

{'provider': 'groq',
 'model': 'openai/gpt-oss-120b',
 'prompt_version': 'v3',
 'validation_version': 'v3',
 'selected': 12,
 'attempted': 1,
 'generated': 1,
 'generation_failures': 0,
 'not_attempted': 11,
 'automatic_passes': 0,
 'automatic_rejections': 1,
 'human_accepted': 0,
 'human_review_status': 'not_reviewed',
 'rate_limit_responses': 0,
 'generation_requests': 1,
 'smoke_status': 'failed',
 'stop_reason': 'Smoke candidate failed full v3 validation; remaining 11 not generated.',
 'completed_all_12': False,
 'assigned_mechanism_distribution': {'sensitive_information': 2,
  'unusual_financial_arrangements': 2,
  'suspicious_application': 2,
  'urgency_pressure': 2,
  'payment_related': 2,
  'implausible_recruitment': 2},
 'generated_mechanism_distribution': {'sensitive_information': 1},
 'passing_mechanism_distribution': {}}

,source_row_id,synthetic_id,validation_version,structural_valid,role_preserved,responsibilities_preserved,qualifications_preserved,location_preserved,unsupported_facts_clear,research_artifacts_clear,...,validation_status,reasons,artifact_source_exemptions,source_jaccard,description_sequence_ratio,fraud_evidence,fraud_sentence_count,human_review_status,semantic_fidelity_verified_by_human,limitations
0,11662,v3_c2f358e6e49f5fb0a6eff2af7d427984,v3,True,True,True,True,True,False,True,...,auto_reject,"[""Unsupported experience duration: 16-18 years""]",[],0.3632,0.5964,"[""government-issued id"", ""bank""]",2,not_reviewed,False,Source anchors and fact patterns are screening...


### Source/Generated Comparison and Human Review

The review template includes all twelve selected sources, explicitly marking unattempted rows. Review fields remain blank. Automatic passes are not human approvals.

In [5]:
for raw in (v3_output / 'regenerated_records.jsonl').read_text(encoding='utf-8').splitlines():
    record = json.loads(raw)
    print('Source:', record['source_row_id'], '|', record['fraud_mechanism'], '|', record['validation_status'])
    display(pd.DataFrame({'source': record['source'], 'generated': record['generated'] or {}}))
    display(record['validation'])
print('Human review:', v3_output / 'review_template.csv')

Source: 11662 | sensitive_information | auto_reject


,source,generated
title,Southend-on-Sea Traineeships Under NAS 16-18 Y...,Customer Service Traineeship – 4‑Week Program ...
company_profile,Established on the principles that full time e...,We are a team of passionate consultants dedica...
description,Government funding is only available for 16-18...,Government funding is available for 16‑18 year...
requirements,16-18 year olds only due to government funding...,"Applicants must be 16‑18 years old, career‑dri..."
benefits,Career prospects.,"Career prospects, a clear pathway to an appren..."


{'validation_version': 'v3',
 'structural_valid': True,
 'role_preserved': True,
 'responsibilities_preserved': True,
 'qualifications_preserved': True,
 'location_preserved': True,
 'unsupported_facts_clear': False,
 'research_artifacts_clear': True,
 'ai_artifacts_clear': True,
 'sensitive_values_clear': True,
 'fraud_intent_present': True,
 'fraud_integrated': True,
 'source_similarity_ok': True,
 'automatic_passed': False,
 'validation_status': 'auto_reject',
 'reasons': ['Unsupported experience duration: 16-18 years'],
 'artifact_source_exemptions': [],
 'source_jaccard': 0.3632,
 'description_sequence_ratio': 0.5964,
 'fraud_evidence': ['government-issued id', 'bank'],
 'fraud_sentence_count': 2,
 'human_review_status': 'not_reviewed',
 'semantic_fidelity_verified_by_human': False,
 'limitations': 'Source anchors and fact patterns are screening evidence, not exhaustive semantic verification.'}

Human review: C:\Users\Rajesh Singh\Desktop\Fraud Detection\AI_Job_Fraud_Detection\data\synthetic\pilot\groq_gpt_v3\review_template.csv


### Limitations and Next Decision

Twelve purposively selected revisions are not a representative dataset. Fixed seeds and saved prompts do not guarantee bit-identical API responses. Generator bias, validator blind spots and source-related leakage remain. Mechanism labels describe assigned intent, not verified fraud truth. Human reviewers must assess coherence, realism, context, unsupported additions and identity/artifact risks. If the smoke test fails, review its evidence and approve a separately versioned change before another attempt. No scaling or Llama experiment is authorized by this notebook.

**Observed result:** one actual Groq generation for source 11662, valid structured output, zero rate-limit/API failures, zero automatic passes and one automatic rejection; eleven sources were not attempted. The sole rejection (`Unsupported experience duration: 16-18 years`) is a validator false positive: source age eligibility `16-18 year olds` was mistaken for professional experience when rephrased as `16-18 years old`. Other screens passed, but this is not human approval. The executed validator and result remain frozen. An offline expected-failure regression documents the defect. Approve an age-aware, separately versioned correction and follow-up protocol before any more generation. Review source-supported training/support rephrased into benefits and employer generalization as well.

## Archived v2 Pilot (Execution Disabled)

# Phase 2B-A: REAL GROQ GPT PILOT

This notebook is a pilot and does not create the final synthetic dataset.
Exactly 20 legitimate sources, seed 42. No classifier, splits or Llama generation.

## MOCK PILOT (preserved)
Historical mock results below are read only. Real API outputs use a separate model-specific directory.

In [6]:
if False:  # Frozen v2 execution: historical code only.
    from pathlib import Path
    import json
    import os
    import sys
    import pandas as pd
    from IPython.display import display
    ROOT = Path.cwd().resolve()
    if not (ROOT / 'data/processed/emscad_clean.csv').exists():
        ROOT = ROOT.parent
    sys.path.insert(0, str(ROOT))
    from src.groq_pilot import GroqPilot
    mock_summary = ROOT / 'results/tables/ai_fraud_pilot_summary.csv'
    if mock_summary.exists():
        display(pd.read_csv(mock_summary))


## 1. Configuration

Set GENERATION_PROVIDER=groq and GENERATION_MODEL in the environment or ignored local .env. Default: openai/gpt-oss-120b. The same procedure can later run an approved Llama comparison by changing only GENERATION_MODEL. Each model gets a separate directory.

Fixed settings: 20 sources, seed 42, temperature 0.7, maximum 4,096 completion tokens, sequential requests, no SDK retries. Seed does not guarantee identical provider responses.

In [7]:
if False:  # Frozen v2 execution: historical code only.
    generation_mode = 'api'
    provider_name = 'groq'
    pilot = GroqPilot(ROOT)
    print('REAL GROQ GPT PILOT')
    print('Mode:', generation_mode, '| Provider:', provider_name, '| Model:', pilot.model)
    print('Output:', pilot.output.relative_to(ROOT))


## 2. Environment / API-Key Check

Local .env is loaded without overriding existing environment variables. Credential values are never displayed.

In [8]:
if False:  # Frozen v2 execution: historical code only.
    print('GROQ_API_KEY configured:', bool(os.getenv('GROQ_API_KEY')))


## 3. Model Availability Check

Query the account's models before generation. GPT-OSS models use strict JSON schema; unsupported schema models use JSON-object mode followed by identical local validation. Record this format difference in later comparisons.

In [9]:
if False:  # Frozen v2 execution: historical code only.
    model_ready = pilot.check_models()
    print('Model availability check:', model_ready)
    print('Available GPT-OSS:', [m for m in pilot.available_models if 'gpt-oss' in m])
    if pilot.problem:
        print('Stopped:', pilot.problem)


## 4. Deterministic Source Selection

Reuse the original selection function and verify identical source IDs against the saved mock selection. Require verified legitimate label f and distinct canonical text groups. Shared descriptions and broader near duplicates still need future split-group review.

In [10]:
if False:  # Frozen v2 execution: historical code only.
    sources = pilot.sources
    assert len(sources) == 20
    assert sources['fraudulent'].eq('f').all()
    assert sources['text_group_id'].is_unique
    display(sources[['source_row_id', 'text_group_id', 'title', 'selection_context', 'selection_length_bucket']])


## 5. Prompt Loading and Version

Prompt v2 retains the occupation and recruitment context and requires an unnamed employer, no personal information, credentials, real contact endpoints or operational payment instructions. Source JSON is data, never instructions. The frozen template, hash, exact request prompts and code/input hashes are saved. Prompt v1 remains unchanged.

In [11]:
if False:  # Frozen v2 execution: historical code only.
    print('Prompt version:', pilot.manifest['prompt_version'])
    print('Validation version:', pilot.manifest['validation_version'])
    print('Prompt SHA-256:', pilot.manifest['prompt_sha256'])
    print(pilot.prompt)


## 6. One-Record API Smoke Test

The first source counts toward the 20. It receives exactly one generation request; API failure, refusal, truncation, malformed JSON or invalid fields stop the run, including reruns. A saved successful smoke response is reused.

In [12]:
if False:  # Frozen v2 execution: historical code only.
    smoke_succeeded = pilot.smoke_test() if model_ready else False
    print('Smoke test:', pilot.smoke_status)
    if pilot.problem:
        print('Stopped:', pilot.problem)
    if pilot.records:
        first = pilot.records[0]
        print('Source:', first['source_row_id'], '|', first['generation_status'])
        if first['generated']:
            display(pd.DataFrame([first['generated']]))


## 7. Complete the 20-Record Pilot

Only a structurally valid smoke result permits the remaining 19 requests. Later sources get at most two attempts, retrying only transient HTTP errors. Rejected candidates are retained. Checkpoints prevent reissuing completed or interrupted requests with unknown server outcomes.

In [13]:
if False:  # Frozen v2 execution: historical code only.
    if smoke_succeeded:
        pilot.generate_remaining()
    else:
        print('The remaining 19 requests were not issued.')
    summary = pilot.export()
    print('Attempted:', summary['number_attempted'], '| Generated:', summary['number_successfully_generated'])


## 8. Automatic Validation

Reuse structural, lexical-context, multi-signal fraud, repetition, artifact and copying checks under validation v2. Low lexical overlap or insufficient fraud-rule evidence routes to human review. High similarity alone never qualifies a candidate and cannot prove semantic preservation. Structural/quality failures receive auto_reject.

An auto_pass is only a screening result; validation_status remains needs_human_review. Named-organization and privacy detection still require manual inspection. No LLM validator calls are made.

In [14]:
if False:  # Frozen v2 execution: historical code only.
    validation_table = pd.read_csv(pilot.output / 'validation_results.csv', keep_default_na=False)
    display(validation_table[['source_row_id', 'generation_status', 'structural_valid', 'context_preserved', 'fraud_intent_present', 'quality_ok', 'automatic_result', 'validation_status', 'validation_reasons']])


## 9. Human-Review Export and Source/Generated Pairs

All 20 sources are included in the review table, even unattempted/failed cases with explicit status. Full source/generated text and blank reviewer fields are exported. Reviewer notes survive reruns. The validated CSV includes only automatic passes awaiting human review. Inspect automatic rejections too.

In [15]:
if False:  # Frozen v2 execution: historical code only.
    review = pd.read_csv(pilot.output / 'human_review_template.csv', keep_default_na=False)
    print('Human-review rows:', len(review))
    for record in pilot.records:
        print('\nSource row:', record['source_row_id'], '|', record['validation_status'])
        generated = record['generated'] or {}
        with pd.option_context('display.max_colwidth', None):
            display(pd.DataFrame([{'field': field, 'source': record['source'].get(field, ''), 'generated': generated.get(field, '')}
                                  for field in ['title', 'company_profile', 'description', 'requirements', 'benefits']]))
        details = record['validation'].get('validation_details', {})
        print('Fraud signals:', details.get('fraud_intent', {}).get('matched_categories', []))
        print('Automatic result:', record['validation'].get('automatic_result'))


## 10. Pilot Summary

Selection, attempts, successful generation and failures are separate counts. Character averages use the five joined fields; generated averages exclude failures. Automatic passes are not human approval.

In [16]:
if False:  # Frozen v2 execution: historical code only.
    display(pd.DataFrame(summary.items(), columns=['metric', 'value']))
    print('Errors:', json.loads((pilot.output / 'errors.json').read_text(encoding='utf-8')))
    print('Protected raw, cleaned and mock file SHA-256 checks passed during export.')


## 11. Limitations and Next Decision

A 20-source pilot cannot establish final dataset validity. Lexical overlap does not prove semantic role preservation. Fraud patterns can miss subtle intent or match legitimate statements. Single-generator bias, generic employer wording, sanitization, validator heuristics, source duplication and distribution shift require scrutiny. Exact text groups do not cover every repeated description. Provider seeds/model aliases cannot ensure bit-for-bit reproducibility; archived responses are the audit record.

Human reviewers must assess role/context, fraud intent, coherence, copying, privacy and real-company impersonation for every candidate. Approve the prompt, protocol and review criteria before an equivalent 20-source Llama pilot. Preserve sources and procedure and report output-format differences. No scale-up, final dataset, splitting, training or Phase 3 is performed.

## GPT Pilot Regeneration - Prompt v3.1 Follow-up

This is a separate follow-up to the frozen v3 smoke-test run. It does not overwrite v2 or v3 evidence. The sole implemented validator change distinguishes a source-supported explicit age-eligibility range from a professional-experience requirement. No other automatic acceptance criterion was weakened. This remains a controlled GPT pilot; it does not run Llama, merge datasets, create splits, or train a classifier.

In [17]:
v31_output = ROOT / 'data/synthetic/pilot/groq_gpt_v3_1'
v31_manifest = json.loads((v31_output / 'manifest.json').read_text(encoding='utf-8'))
v31_summary = json.loads((v31_output / 'generation_summary.json').read_text(encoding='utf-8'))
print('Frozen v3.1 artifact - read only')
display({key: v31_manifest[key] for key in ['provider', 'model', 'prompt_version', 'validation_version', 'source_ids', 'excluded_source_id']})
display(pd.DataFrame(v31_manifest['assignments'])[['source_row_id', 'fraud_mechanism']])

Frozen v3.1 artifact - read only


{'provider': 'groq',
 'model': 'openai/gpt-oss-120b',
 'prompt_version': 'v3.1',
 'validation_version': 'v3.1-groq-pilot',
 'source_ids': [11662,
  12066,
  13604,
  14350,
  14484,
  15885,
  3203,
  4581,
  5581,
  6199,
  7192,
  9208],
 'excluded_source_id': 10308}

,source_row_id,fraud_mechanism
0,11662,sensitive_information
1,12066,unusual_financial_arrangements
2,13604,suspicious_application
3,14350,urgency_pressure
4,14484,sensitive_information
5,15885,payment_related
6,3203,unusual_financial_arrangements
7,4581,payment_related
8,5581,implausible_recruitment
9,6199,suspicious_application


### v3.1 Validation and Rate-Limit Controls

The validator still requires strict five-field JSON; non-empty title and description; role, responsibility, qualification and location anchors; fraud evidence integrated into recruitment steps; research/meta-artifact and AI-artifact screening; source-similarity/copy checks; unsupported-fact checks; and sensitive-value screening. It only exempts an age range such as `16-18 years old` when the exact age range already appears in the source as age eligibility. New professional-experience durations, benefits, employment arrangements, compensation and remote-work claims remain rejected.

Generation is sequential. HTTP 429 honors `Retry-After` when supplied, otherwise uses 65-second exponential backoff, with at most three attempts per source and a stop after three consecutive 429 responses. State and exports are saved incrementally.

In [18]:
v31_state = json.loads((v31_output / 'state.json').read_text(encoding='utf-8'))
print('Frozen v3.1 smoke status:', v31_state['smoke_status'])
print('Stop reason:', v31_state['stop_reason'])

Frozen v3.1 smoke status: failed
Stop reason: Smoke candidate failed full v3 validation; remaining 11 not generated.


### Conditional Remaining Regeneration

The remaining eleven requests run only when the full smoke validation passes. An automatic pass is not human approval; every successful candidate is exported with `pending_human_review` and a blank review template.

In [19]:
display(v31_summary)
display(pd.read_csv(v31_output / 'validation_results.csv'))
print('Review template:', v31_output / 'review_template.csv')

{'provider': 'groq',
 'model': 'openai/gpt-oss-120b',
 'prompt_version': 'v3.1',
 'validation_version': 'v3.1-groq-pilot',
 'selected': 12,
 'attempted': 1,
 'generated': 1,
 'generation_failures': 0,
 'not_attempted': 11,
 'automatic_passes': 0,
 'automatic_rejections': 1,
 'human_accepted': 0,
 'human_review_status': 'pending_human_review',
 'rate_limit_responses': 0,
 'generation_requests': 1,
 'smoke_status': 'failed',
 'stop_reason': 'Smoke candidate failed full v3 validation; remaining 11 not generated.',
 'completed_all_12': False,
 'assigned_mechanism_distribution': {'sensitive_information': 2,
  'unusual_financial_arrangements': 2,
  'suspicious_application': 2,
  'urgency_pressure': 2,
  'payment_related': 2,
  'implausible_recruitment': 2},
 'generated_mechanism_distribution': {'sensitive_information': 1},
 'passing_mechanism_distribution': {},
 'validation_lineage': 'v3-groq-pilot',
 'structural_validation_failures': 0,
 'records_requiring_human_review': 1,
 'rate_limit_fai

,source_row_id,synthetic_id,validation_version,structural_valid,role_preserved,responsibilities_preserved,qualifications_preserved,location_preserved,unsupported_facts_clear,research_artifacts_clear,...,validation_status,reasons,artifact_source_exemptions,source_jaccard,description_sequence_ratio,fraud_evidence,fraud_sentence_count,human_review_status,semantic_fidelity_verified_by_human,limitations
0,11662,v3_1_c944091f744d5cbdaa564b06a1374e4a,v3.1-groq-pilot,True,True,True,True,True,True,True,...,auto_reject,"[""Mechanism is confined to one sentence or lac...",[],0.2974,0.5556,"[""identity"", ""bank""]",1,pending_human_review,False,Source anchors and fact patterns are screening...


Review template: C:\Users\Rajesh Singh\Desktop\Fraud Detection\AI_Job_Fraud_Detection\data\synthetic\pilot\groq_gpt_v3_1\review_template.csv


### Interpretation

This run reports generation success, validation decisions, artifacts, unsupported-fact flags and mechanism assignments. It does not claim human validation, model quality, dataset validity or readiness for scaling. Review the saved source/generated pairs before any further model comparison or dataset construction.

## GPT Pilot Regeneration - Prompt v3.2

This is an isolated follow-up to frozen v2, v3 and v3.1 artifacts. v3.1's smoke candidate passed every check except a two-sentence fraud-integration threshold. That threshold was removed because natural fraudulent recruitment signals may be coherent in one sentence. No structural, context, fraud-intent, artifact, unsupported-fact, sensitive-value or similarity check was weakened. No Llama, dataset merge, split or model training is performed here.

In [20]:
v32_output = ROOT / 'data/synthetic/pilot/groq_gpt_v3_2'
v32_manifest = json.loads((v32_output / 'manifest.json').read_text(encoding='utf-8'))
v32_summary = json.loads((v32_output / 'generation_summary.json').read_text(encoding='utf-8'))
print('Frozen v3.2 artifact - read only')
display({key: v32_manifest[key] for key in ['provider', 'model', 'prompt_version', 'validation_version', 'source_ids', 'excluded_source_id']})
display(pd.DataFrame(v32_manifest['assignments'])[['source_row_id', 'fraud_mechanism']])

Frozen v3.2 artifact - read only


{'provider': 'groq',
 'model': 'openai/gpt-oss-120b',
 'prompt_version': 'v3.2',
 'validation_version': 'v3.2-groq-pilot',
 'source_ids': [11662,
  12066,
  13604,
  14350,
  14484,
  15885,
  3203,
  4581,
  5581,
  6199,
  7192,
  9208],
 'excluded_source_id': 10308}

,source_row_id,fraud_mechanism
0,11662,sensitive_information
1,12066,unusual_financial_arrangements
2,13604,suspicious_application
3,14350,urgency_pressure
4,14484,sensitive_information
5,15885,payment_related
6,3203,unusual_financial_arrangements
7,4581,payment_related
8,5581,implausible_recruitment
9,6199,suspicious_application


### Contextual Integration Validation

Fraud evidence must be clear and tied to an application, pre-screening, interview, selection, enrollment or onboarding step. A single sentence can pass when it coherently expresses that relationship. The validator rejects warning-style language, a disconnected final signal that lacks recruitment context, and excessive mechanism repetition. The assigned mechanism still needs its two category-specific indicators and an applicant obligation. This is a transparent heuristic rather than semantic proof; all candidates remain pending independent human review.

In [21]:
v32_state = json.loads((v32_output / 'state.json').read_text(encoding='utf-8'))
print('Frozen v3.2 smoke status:', v32_state['smoke_status'])
print('Stop reason:', v32_state['stop_reason'])

Frozen v3.2 smoke status: failed
Stop reason: Smoke candidate failed full v3 validation; remaining 11 not generated.


### Conditional Remaining Eleven

The remaining requests run sequentially only after the full smoke test passes. HTTP 429 handling honors Retry-After, otherwise applies bounded exponential backoff. Successful and failed attempts are saved incrementally and never silently regenerated.

In [22]:
display(v32_summary)
display(pd.read_csv(v32_output / 'validation_results.csv'))
print('Human review template:', v32_output / 'review_template.csv')

{'provider': 'groq',
 'model': 'openai/gpt-oss-120b',
 'prompt_version': 'v3.2',
 'validation_version': 'v3.2-groq-pilot',
 'selected': 12,
 'attempted': 1,
 'generated': 1,
 'generation_failures': 0,
 'not_attempted': 11,
 'automatic_passes': 0,
 'automatic_rejections': 1,
 'human_accepted': 0,
 'human_review_status': 'pending_human_review',
 'rate_limit_responses': 0,
 'generation_requests': 1,
 'smoke_status': 'failed',
 'stop_reason': 'Smoke candidate failed full v3 validation; remaining 11 not generated.',
 'completed_all_12': False,
 'assigned_mechanism_distribution': {'sensitive_information': 2,
  'unusual_financial_arrangements': 2,
  'suspicious_application': 2,
  'urgency_pressure': 2,
  'payment_related': 2,
  'implausible_recruitment': 2},
 'generated_mechanism_distribution': {'sensitive_information': 1},
 'passing_mechanism_distribution': {},
 'validation_lineage': 'v3.1-groq-pilot',
 'structural_validation_failures': 0,
 'records_requiring_human_review': 1,
 'rate_limit_f

,source_row_id,synthetic_id,validation_version,structural_valid,role_preserved,responsibilities_preserved,qualifications_preserved,location_preserved,unsupported_facts_clear,research_artifacts_clear,...,artifact_source_exemptions,source_jaccard,description_sequence_ratio,fraud_evidence,fraud_sentence_count,human_review_status,semantic_fidelity_verified_by_human,limitations,age_eligibility_exemption,contextual_integration
0,11662,v3_2_5a49ac50cd5c588ab3349bc50b025f19,v3.2-groq-pilot,True,True,True,True,True,False,True,...,[],0.2576,0.5005,"[""government-issued id"", ""bank""]",1,pending_human_review,False,Source anchors and fact patterns are screening...,"[""Unsupported experience duration: 16-18 years""]","{""accepted"": true, ""linked_sentence_count"": 1,..."


Human review template: C:\Users\Rajesh Singh\Desktop\Fraud Detection\AI_Job_Fraud_Detection\data\synthetic\pilot\groq_gpt_v3_2\review_template.csv


### Interpretation

Automatic passes are screening outcomes only. The review template records context preservation, fraud intent, realism, unsupported facts, research/meta artifacts, AI artifacts and overall suitability. No candidate is human-approved in this notebook. Stop after this GPT v3.2 pilot; review the outputs before any other model or phase.

## GPT Pilot Regeneration - Prompt v3.3

This isolated follow-up preserves frozen v2, v3, v3.1 and v3.2 artifacts. The v3.2 smoke candidate passed contextual integration but was rejected for inventing a full-time arrangement. The validator remains strict. Prompt v3.3 treats employment arrangements and working hours as source-bound facts and tells the generator to omit absent information rather than invent it. No Llama generation, merging, splitting, or model training is performed.

### Employment-Arrangement Preservation

The v3.3 validator compares explicit terms in source and output: full-time, part-time, temporary, permanent, contract, internship, traineeship, apprenticeship, volunteer, freelance, seasonal, remote and hybrid. It rejects new terms and source terms that are omitted or replaced, as well as changed weekly-hours claims. Contextual fraud integration remains a recruitment-process coherence test, not a sentence-count rule.

In [ ]:
v33_output = ROOT / 'data/synthetic/pilot/groq_gpt_v3_3'
v33_manifest = json.loads((v33_output / 'manifest.json').read_text(encoding='utf-8'))
v33_summary = json.loads((v33_output / 'generation_summary.json').read_text(encoding='utf-8'))
v33_state = json.loads((v33_output / 'state.json').read_text(encoding='utf-8'))
print('Frozen v3.3 artifact - read only')
display({key: v33_manifest[key] for key in ['provider', 'model', 'prompt_version', 'validation_version', 'source_ids', 'excluded_source_id']})
print('Smoke status:', v33_state['smoke_status'])
print('Stop reason:', v33_state['stop_reason'])
display(v33_summary)
display(pd.read_csv(v33_output / 'validation_results.csv'))
print('Human review template:', v33_output / 'review_template.csv')

### Interpretation

The saved output distinguishes API-generation failures, rate limits and automatic validation rejections. Every generated candidate remains pending human review. This notebook is an audit of completed versioned runs; it does not issue generation requests when executed.